In [1]:
import geopandas as gpd
from shapely.geometry import Point

from pyproj.aoi import AreaOfInterest
from pyproj.database import query_utm_crs_info
import pdal
import pandas as pd

(PDAL Error) Can't load library /Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libpdal_plugin_reader_hdf.dylib: Failed to load "/Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libpdal_plugin_reader_hdf.dylib": dlopen(/Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libpdal_plugin_reader_hdf.dylib, 0x0002): Library not loaded: @rpath/libhdf5_cpp.320.dylib
  Referenced from: <3556841C-EDE8-3560-B617-381AF0E87C21> /Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libpdal_plugin_reader_hdf.20.2.0.dylib
  Reason: tried: '/Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libhdf5_cpp.320.dylib' (no such file), '/Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libhdf5_cpp.320.dylib' (no such file), '/Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/libhdf5_cpp.320.dylib' (no such file), '/Users/sala6791/PDAL/.pixi/envs/goldendoodle/bin/../lib/libhdf5_cpp.320.dylib' (no such file), '/usr/local/lib/libhdf5_cpp.320.dylib' (no such file), '/usr/lib/libhdf5_cpp.320.dylib' (no such file, not in dyld

# metadata table of available USGS 3DEP 

In [ ]:
# resources = gpd.read_file(
#     "https://raw.githubusercontent.com/hobu/usgs-lidar/master/boundaries/resources.geojson" # updated frequently
#     # "https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/metadata/WESM.gpkg" # hangs
# )
# # merge with csv with additional metadata
# wesm_metadata = pd.read_csv('./WESM.csv',parse_dates=['collect_start','collect_end'])  # from https://rockyweb.usgs.gov/vdelivery/Datasets/Staged/Elevation/metadata/WESM.csv

# resources = resources.merge(wesm_metadata, how='left', left_on='name', right_on='workunit'  )
# resources['collection_year']=resources['collect_start'].apply(lambda x:x.year)
# resources.to_file('usgs_3dep_resources.geojson', driver='GeoJSON')

In [28]:
# jupyter notebook cell to draw bbox on a map and capture it in a variable
from ipyleaflet import DrawControl, Map, basemaps

# 1. Initialize the interactive map
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(40.015, -105.270),  # Latitude, Longitude center
    zoom=10,
)

# 2. Define a global variable to store your bounding box coordinates
bbox_coords = None


# 3. Define a callback function to handle draw events
def handle_draw(target, action, geo_json):
    global bbox_coords
    # Check if the drawn shape is a Polygon/Rectangle
    if geo_json["geometry"]["type"] == "Polygon" and action == "created":
        # Extract the coordinate ring (outer boundary)
        coordinates = geo_json["geometry"]["coordinates"][0]

        # Calculate bounding box bounds: [min_lon, min_lat, max_lon, max_lat]
        lons = [pt[0] for pt in coordinates]
        lats = [pt[1] for pt in coordinates]
        bbox_coords = [min(lons), min(lats), max(lons), max(lats)]

        print(f"Captured BBOX [min_lon, min_lat, max_lon, max_lat]:")
        print(bbox_coords)


# 4. Configure the DrawControl (keep only the rectangle tool active)
draw_control = DrawControl()
draw_control.polyline = {}
draw_control.polygon = {}
draw_control.circlemarker = {}
draw_control.rectangle = {
    "shapeOptions": {"fillColor": "#3388ff", "color": "#3388ff", "fillOpacity": 0.2}
}

# Attach the callback function to the draw control
draw_control.on_draw(handle_draw)

# Add the drawing tools to the map
m.add(draw_control)

# Display the map widget
m

Map(center=[40.015, -105.27], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoo…

In [30]:
bbox_center = [
    (bbox_coords[0] + bbox_coords[2]) / 2,
    (bbox_coords[1] + bbox_coords[3]) / 2,
]


In [31]:
pt = Point(bbox_center[0], bbox_center[1])
matches = resources[resources.geometry.contains(pt)]
matches.sort_values(by='collect_start', ascending=False)

,name,id,count,url,geometry,workunit,workunit_id,project,project_id,collect_start,...,sourcedem_category,sourcedem_reason,onemeter_category,onemeter_reason,seamless_category,seamless_reason,lpc_link,sourcedem_link,metadata_link,collection_year
273,CO_DRCOG_3_2020,273,58382624205,https://s3-us-west-2.amazonaws.com/usgs-lidar-...,"MULTIPOLYGON (((-105.71946 39.54404, -105.6655...",CO_DRCOG_3_2020,219761.0,CO_DRCOG_2020_B20,193312.0,2020-06-14,...,Meets,Meets 3DEP source DEM requirements,Meets,Meets 3DEP 1-m DEM requirements,Meets,Meets 3DEP seamless DEM requirements,https://rockyweb.usgs.gov/vdelivery/Datasets/S...,https://prd-tnm.s3.amazonaws.com/index.html?pr...,https://prd-tnm.s3.amazonaws.com/index.html?pr...,2020.0


In [42]:
import requests
meta = requests.get(matches["metadata_link"].values[0])
print(meta)  # per-file source list, sometimes has per-tile dates

<Response [200]>


In [32]:
matches['collection_year']=matches['collect_start'].apply(lambda x:x.year)

/Users/sala6791/PDAL/.pixi/envs/goldendoodle/lib/python3.13/site-packages/geopandas/geodataframe.py:1969: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [33]:
import pdal, json

bounds = {
    "minx": bbox_coords[0],
    "miny": bbox_coords[1],
    "maxx": bbox_coords[2],
    "maxy": bbox_coords[3],
    "minz": 0,
    "maxz": 5000,   # meters; wide enough to not clip real elevations
    "crs": "EPSG:4326",
}

pipeline = {
    "pipeline": [
        {
            "type": "readers.ept",
            "filename": matches["url"].values[0],
            "bounds": bounds,
        },
        # {"type": "writers.las", "filename": "clip.laz"},
        {"type": "writers.copc", "filename": "clip.copc.laz"},

    ]
}

pdal.Pipeline(json.dumps(pipeline)).execute()

24848278

In [20]:
import pdal, json
pipeline_def = {
    "pipeline": [
        {
            "type": "readers.ept",
            "filename": matches["url"].values[0],
            "bounds": bounds,
        }
    ]
}

pipeline = pdal.Pipeline(json.dumps(pipeline_def))
pipeline.execute()

arr = pipeline.arrays[0]   # structured numpy array: fields like 'X','Y','Z','Intensity','Classification', ...
print(arr.shape, arr.dtype.names)


(9258501,) ('Classification', 'EdgeOfFlightLine', 'GpsTime', 'Intensity', 'KeyPoint', 'NumberOfReturns', 'Overlap', 'PointSourceId', 'ReturnNumber', 'ScanAngleRank', 'ScanChannel', 'ScanDirectionFlag', 'Synthetic', 'UserData', 'Withheld', 'X', 'Y', 'Z')


In [25]:
ASPRS_CLASSES = {
    0:  "Created, never classified",
    1:  "Unclassified",
    2:  "Ground",
    3:  "Low Vegetation",
    4:  "Medium Vegetation",
    5:  "High Vegetation",
    6:  "Building",
    7:  "Low Point (noise)",
    8:  "Reserved",              # was Model Key-point in LAS 1.1-1.3
    9:  "Water",
    10: "Rail",
    11: "Road Surface",
    12: "Reserved",              # was Overlap Points in LAS 1.1-1.3
    13: "Wire - Guard (Shield)",
    14: "Wire - Conductor (Phase)",
    15: "Transmission Tower",
    16: "Wire-Structure Connector",
    17: "Bridge Deck",
    18: "High Noise",
    19: "Overhead Structure",
    20: "Ignored Ground",
    21: "Snow",
    22: "Temporal Exclusion",
    # 23-63 reserved, 64-255 user definable
}

In [30]:
import laspy
las = laspy.read('clip.copc.laz')


In [35]:
las.header.min_gps_time

0.0